In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from io import BytesIO
import pandas as pd
import torch
import requests
import os

# === Load BLIP model ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
model.eval()

@torch.no_grad()
def caption_from_url(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = processor(img, return_tensors="pt").to(device)
        output = model.generate(**inputs)
        caption = processor.decode(output[0], skip_special_tokens=True)
        return caption
    except Exception:
        return None  # If any error, return None so fallback caption can be applied

# === Load dataset ===
input_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/full-image.csv'
output_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv'

df = pd.read_csv(input_path)

# Ensure required columns exist
if 'caption' not in df.columns:
    df['caption'] = None
if 'image_exists' not in df.columns:
    df['image_exists'] = None

# Filter rows needing processing
df_to_process = df[df['caption'].isna() | (df['caption'] == '')].copy()
print(f"🧠 Total missing captions: {len(df_to_process)}")

# Reset output file
open(output_path, 'w').close()

# === Helper: Check if image exists (via HEAD request) ===
def check_image_exists(url):
    try:
        if pd.isna(url) or not isinstance(url, str) or url.strip() == '':
            return False
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.head(url, timeout=5, headers=headers, allow_redirects=True)
        return response.status_code == 200 and 'image' in response.headers.get('Content-Type', '')
    except:
        return False

# === Process in batches and save ===
batch_size = 500
total = len(df_to_process)

for start in range(0, total, batch_size):
    end = min(start + batch_size, total)
    batch = df_to_process.iloc[start:end].copy()
    print(f"🔄 Processing batch {start}-{end}")

    for idx in batch.index:
        url = batch.at[idx, 'image_url']

        exists = check_image_exists(url)
        batch.at[idx, 'image_exists'] = exists

        if exists:
            caption = caption_from_url(url)
            batch.at[idx, 'caption'] = caption if caption else "Error generating caption"
        else:
            batch.at[idx, 'caption'] = "Image does not exist"

    # Append batch to output CSV
    batch.to_csv(output_path, mode='a', index=False, header=(start == 0))
    print(f"✅ Appended rows {start}-{end} to: {output_path}")


🧠 Total missing captions: 91323
🔄 Processing batch 0-500
✅ Appended rows 0-500 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 500-1000


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Appended rows 500-1000 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 1000-1500
✅ Appended rows 1000-1500 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 1500-2000
✅ Appended rows 1500-2000 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 2000-2500
✅ Appended rows 2000-2500 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 2500-3000
✅ Appended rows 2500-3000 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 3000-3500
✅ Appended rows 

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from io import BytesIO
import pandas as pd
import torch
import requests
import os

# === Load BLIP model ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
model.eval()

@torch.no_grad()
def caption_from_url(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = processor(img, return_tensors="pt").to(device)
        output = model.generate(**inputs)
        caption = processor.decode(output[0], skip_special_tokens=True)
        return caption
    except Exception:
        return None  # If any error, return None so fallback caption can be applied

# === Load dataset ===
input_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/full-image.csv'
output_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv'

df = pd.read_csv(input_path)

# Ensure required columns exist
if 'caption' not in df.columns:
    df['caption'] = None
if 'image_exists' not in df.columns:
    df['image_exists'] = None

# Filter rows needing processing
df_to_process = df[df['caption'].isna() | (df['caption'] == '')].copy()
print(f"🧠 Total missing captions: {len(df_to_process)}")

# Reset output file
open(output_path, 'w').close()

# === Helper: Check if image exists (via HEAD request) ===
def check_image_exists(url):
    try:
        if pd.isna(url) or not isinstance(url, str) or url.strip() == '':
            return False
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.head(url, timeout=5, headers=headers, allow_redirects=True)
        return response.status_code == 200 and 'image' in response.headers.get('Content-Type', '')
    except:
        return False

# === Process in batches and save ===
batch_size = 500
total = len(df_to_process)

for start in range(48500, total, batch_size):
    end = min(start + batch_size, total)
    batch = df_to_process.iloc[start:end].copy()
    print(f"🔄 Processing batch {start}-{end}")

    for idx in batch.index:
        url = batch.at[idx, 'image_url']

        exists = check_image_exists(url)
        batch.at[idx, 'image_exists'] = exists

        if exists:
            caption = caption_from_url(url)
            batch.at[idx, 'caption'] = caption if caption else "Error generating caption"
        else:
            batch.at[idx, 'caption'] = "Image does not exist"

    # Append batch to output CSV
    batch.to_csv(output_path, mode='a', index=False, header=(start == 0))
    print(f"✅ Appended rows {start}-{end} to: {output_path}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

🧠 Total missing captions: 91323
🔄 Processing batch 48500-49000


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Appended rows 48500-49000 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 49000-49500
✅ Appended rows 49000-49500 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 49500-50000
✅ Appended rows 49500-50000 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 50000-50500
✅ Appended rows 50000-50500 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 50500-51000
✅ Appended rows 50500-51000 to: /content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv
🔄 Processing batch 51000-5

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from io import BytesIO
import pandas as pd
import torch
import requests
import os

# === Load BLIP model ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
model.eval()

@torch.no_grad()
def caption_from_url(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = processor(img, return_tensors="pt").to(device)
        output = model.generate(**inputs)
        caption = processor.decode(output[0], skip_special_tokens=True)
        return caption
    except Exception:
        return None  # If any error, return None so fallback caption can be applied

# === Load dataset ===
input_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/full-image.csv'
output_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/caption-combined-dataset.csv'

df = pd.read_csv(input_path)

# Ensure required columns exist
if 'caption' not in df.columns:
    df['caption'] = None
if 'image_exists' not in df.columns:
    df['image_exists'] = None

# Filter rows needing processing
df_to_process = df[df['caption'].isna() | (df['caption'] == '')].copy()
print(f"🧠 Total missing captions: {len(df_to_process)}")

# Reset output file
open(output_path, 'w').close()

# === Helper: Check if image exists (via HEAD request) ===
def check_image_exists(url):
    try:
        if pd.isna(url) or not isinstance(url, str) or url.strip() == '':
            return False
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.head(url, timeout=5, headers=headers, allow_redirects=True)
        return response.status_code == 200 and 'image' in response.headers.get('Content-Type', '')
    except:
        return False

# === Process in batches and save ===
batch_size = 500
total = len(df_to_process)

for start in range(73000, total, batch_size):
    end = min(start + batch_size, total)
    batch = df_to_process.iloc[start:end].copy()
    print(f"🔄 Processing batch {start}-{end}")

    for idx in batch.index:
        url = batch.at[idx, 'image_url']

        exists = check_image_exists(url)
        batch.at[idx, 'image_exists'] = exists

        if exists:
            caption = caption_from_url(url)
            batch.at[idx, 'caption'] = caption if caption else "Error generating caption"
        else:
            batch.at[idx, 'caption'] = "Image does not exist"

    # Append batch to output CSV
    batch.to_csv(output_path, mode='a', index=False, header=(start == 0))
    print(f"✅ Appended rows {start}-{end} to: {output_path}")
